In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "notebooks").exists():
    ROOT = ROOT.parent

sys.path.append(str(ROOT / "notebooks"))

from utils.financebench_eval import (
    get_doc_names,
    get_doc_names_from_pdfs,
    load_env_file,
    load_financebench_rows,
)
from utils.financebench_grid import default_configs, run_retrieval_eval

load_env_file(ROOT / "backend/.env")

FINANCEBENCH_ROOT = ROOT / "notebooks/financebench"
QUESTIONS_PATH = FINANCEBENCH_ROOT / "data" / "financebench_open_source.jsonl"
PDF_DIR = FINANCEBENCH_ROOT / "pdfs"

MAX_QUESTIONS = 50
USE_FULL_CORPUS = False
EMBEDDING_MODEL = "text-embedding-3-small"
EMBEDDING_BATCH_SIZE = 32
RUN_LABEL = f"page_only_{'full' if USE_FULL_CORPUS else 'sample'}_q{MAX_QUESTIONS or 'all'}_{EMBEDDING_MODEL.replace('-', '_')}"
CHECKPOINT_PATH = FINANCEBENCH_ROOT / f"eval_checkpoint_{RUN_LABEL}.json"

/home/hoangvu/Coursework_Y4/DataFlatform/final-project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
rows = load_financebench_rows(QUESTIONS_PATH, max_questions=MAX_QUESTIONS)
doc_names = get_doc_names_from_pdfs(PDF_DIR) if USE_FULL_CORPUS else get_doc_names(rows)
configs = default_configs()

len(rows), len(doc_names), configs, CHECKPOINT_PATH.name

(50,
 39,
 [{'retrieval': 'dense'}, {'retrieval': 'hybrid'}],
 'eval_checkpoint_page_only_sample_q50_text_embedding_3_small.json')

In [3]:
results, details = run_retrieval_eval(
    rows,
    pdf_dir=PDF_DIR,
    doc_names=doc_names,
    embedding_model=EMBEDDING_MODEL,
    configs=configs,
    embedding_batch_size=EMBEDDING_BATCH_SIZE,
    checkpoint_path=CHECKPOINT_PATH,
    resume=True,
)

results

configs:   0%|          | 0/2 [00:00<?, ?config/s]

Start config: page | dense | no_rerank


configs:  50%|█████     | 1/2 [07:48<07:48, 468.06s/config]

Done config: page | dense | no_rerank in 7m 48s
Start config: page | hybrid | no_rerank


configs: 100%|██████████| 2/2 [08:04<00:00, 242.01s/config]


Done config: page | hybrid | no_rerank in 16s


[{'config': 'page | dense | no_rerank',
  'chunking': 'page',
  'retrieval': 'dense',
  'ranking': 'no_rerank',
  'questions': 50,
  'page_hit@1': 0.14,
  'page_hit@5': 0.4,
  'page_hit@20': 0.68,
  'page_mrr@20': 0.2618654658035463,
  'fail_at_5': 30,
  'not_in_top_20': 16,
  'in_top_20_but_not_top_5': 14,
  'ranking_problem_ratio': 0.28,
  'retrieval_problem_ratio': 0.32},
 {'config': 'page | hybrid | no_rerank',
  'chunking': 'page',
  'retrieval': 'hybrid',
  'ranking': 'no_rerank',
  'questions': 50,
  'page_hit@1': 0.12,
  'page_hit@5': 0.24,
  'page_hit@20': 0.58,
  'page_mrr@20': 0.20988254738678308,
  'fail_at_5': 38,
  'not_in_top_20': 21,
  'in_top_20_but_not_top_5': 17,
  'ranking_problem_ratio': 0.34,
  'retrieval_problem_ratio': 0.42}]

In [4]:
details["page | dense | no_rerank"]["summary"]

{'questions': 50,
 'page_hit@1': 0.14,
 'page_hit@5': 0.4,
 'page_hit@20': 0.68,
 'page_mrr@20': 0.2618654658035463,
 'fail_at_5': 30,
 'not_in_top_20': 16,
 'in_top_20_but_not_top_5': 14}

In [ ]:
details["page | hybrid | no_rerank"]["summary"]

{'questions': 50,
 'page_hit@1': 0.12,
 'page_hit@5': 0.24,
 'page_hit@20': 0.58,
 'page_mrr@20': 0.20988254738678308,
 'fail_at_5': 38,
 'not_in_top_20': 21,
 'in_top_20_but_not_top_5': 17}